# 05 — Matrix Multiplication as a Systems Problem

**Master syllabus coverage:** V2 1.5

## Why this matters

Matmul dominates Transformer compute, yet the three-loop equation says little about real performance. Loop order, tiling, packing, vectorization, precision, and shape-specific dispatch determine hardware efficiency.

## Learning objectives

- Implement and compare two loop orders for matrix multiplication.
- Implement boundary-safe tiled multiplication.
- Verify numerical error across square and rectangular shapes.
- Identify the major Transformer matmul shape families and their operation counts.

Work through the notebook in order. Predict shapes and results before running each code cell. An assertion failure is part of the lesson: read it before changing code.


## 1. Same equation, different execution orders

For $A=[M,K]$, $B=[K,N]$, output $C=[M,N]$ uses $2MKN$ approximate floating-point
operations. Six loop permutations are mathematically equivalent but access rows and
columns differently. Optimized GEMM implementations also vectorize, parallelize, pack,
tile, and select hardware-specific instructions.


In [ ]:
import time
import numpy as np
import torch

def matmul_ijk(a: np.ndarray, b: np.ndarray) -> np.ndarray:
    m, k = a.shape; k2, n = b.shape
    assert k == k2
    out = np.zeros((m, n), dtype=np.float32)
    for i in range(m):
        for j in range(n):
            for p in range(k):
                out[i, j] += a[i, p] * b[p, j]
    return out

def matmul_ikj(a: np.ndarray, b: np.ndarray) -> np.ndarray:
    m, k = a.shape; k2, n = b.shape
    assert k == k2
    out = np.zeros((m, n), dtype=np.float32)
    for i in range(m):
        for p in range(k):
            a_value = a[i, p]
            for j in range(n):
                out[i, j] += a_value * b[p, j]
    return out

rng = np.random.default_rng(42)
a = rng.normal(size=(48, 48)).astype(np.float32)
b = rng.normal(size=(48, 48)).astype(np.float32)
reference = a @ b
for name, function in (("ijk", matmul_ijk), ("ikj", matmul_ikj)):
    start = time.perf_counter(); result = function(a, b); elapsed = time.perf_counter() - start
    np.testing.assert_allclose(result, reference, rtol=2e-5, atol=2e-5)
    print(name, f"{elapsed*1e3:.1f} ms")


## 2. Tiling increases reuse in faster memory

A tiled algorithm operates on submatrices so pieces of A and B can be reused from cache
or GPU shared/local memory before eviction. Tile size must fit the hierarchy and execution
resources. The Python implementation exposes structure but cannot demonstrate the full
benefit of a compiled vectorized kernel.


In [ ]:
def tiled_matmul(a: np.ndarray, b: np.ndarray, tile: int) -> np.ndarray:
    m, k = a.shape; k2, n = b.shape
    assert k == k2
    out = np.zeros((m, n), dtype=np.float32)
    for i0 in range(0, m, tile):
        for p0 in range(0, k, tile):
            for j0 in range(0, n, tile):
                i1, p1, j1 = min(i0 + tile, m), min(p0 + tile, k), min(j0 + tile, n)
                out[i0:i1, j0:j1] += a[i0:i1, p0:p1] @ b[p0:p1, j0:j1]
    return out

rectangular_a = rng.normal(size=(37, 53)).astype(np.float32)
rectangular_b = rng.normal(size=(53, 29)).astype(np.float32)
for tile in (4, 8, 16, 32):
    result = tiled_matmul(rectangular_a, rectangular_b, tile)
    np.testing.assert_allclose(result, rectangular_a @ rectangular_b, rtol=2e-5, atol=2e-5)
print("tiled rectangular multiplication passed boundary tiles")


## 3. Verification includes tolerance and shape families

Floating-point accumulation order changes small rounding details. Test maximum absolute
and relative error with precision-appropriate tolerances. Include non-square matrices,
dimensions not divisible by tile size, zeros, large/small magnitudes, and incompatible
shapes.


In [ ]:
result = tiled_matmul(rectangular_a, rectangular_b, tile=16)
expected = rectangular_a @ rectangular_b
absolute = np.max(np.abs(result - expected))
relative = np.max(np.abs(result - expected) / np.maximum(np.abs(expected), 1e-6))
print("output shape:", result.shape, "max abs error:", absolute, "max relative error:", relative)
assert result.shape == (37, 29)


## 4. Transformer matmuls have varied shapes

Embedding-width projections, QKᵀ, attention-value products, MLP expansion/contraction,
and LM heads are rectangular and batched. Kernel choice depends on all dimensions, batch,
dtype, layout, and device—not just total FLOPs.


In [ ]:
B, T, C, H, D, V = 4, 128, 256, 8, 32, 20_000
workloads = {
    "QKV projection": ((B * T, C), (C, 3 * C)),
    "one-head QKᵀ": ((T, D), (D, T)),
    "MLP expansion": ((B * T, C), (C, 4 * C)),
    "LM head": ((B * T, C), (C, V)),
}
for name, (left_shape, right_shape) in workloads.items():
    operations = 2 * left_shape[0] * left_shape[1] * right_shape[1]
    print(f"{name:16}: {left_shape} @ {right_shape}, ≈{operations/1e6:.1f} MFLOP")


## 5. Optimized libraries are algorithm selectors

NumPy BLAS and PyTorch dispatch to tuned implementations. A fair comparison separates
setup, warms up, repeats, synchronizes accelerators, and checks output. This benchmark is
a local observation, not evidence that one library wins on all hardware.


In [ ]:
large_a = torch.randn(256, 384)
large_b = torch.randn(384, 192)
for _ in range(3): _ = large_a @ large_b
start = time.perf_counter()
for _ in range(20): optimized = large_a @ large_b
elapsed = (time.perf_counter() - start) / 20
print(f"PyTorch {tuple(large_a.shape)} @ {tuple(large_b.shape)}: {elapsed*1e3:.3f} ms")
assert optimized.shape == (256, 192)


## Failure modes to recognize

- **Poor loop locality:** mathematically identical code repeatedly reloads data.
- **Broken boundary tile:** non-divisible dimensions are skipped or accessed out of range.
- **Exact-equality test:** valid floating-point reorderings fail an inappropriate comparison.
- **Square-only benchmark:** conclusions do not transfer to Transformer projections.

These are diagnostic patterns, not trivia. When a future model fails, reduce the problem to the smallest example in this notebook and test the relevant invariant.


## Deliberate practice

1. Implement and time a third loop order while explaining its access pattern.
2. Plot timing across tile sizes for two rectangular shapes without claiming a universal best tile.
3. Estimate operations and optimistic bytes moved for QKV, MLP, and LM-head projections.

Do not merely rerun the provided cells. Make a prediction, change one variable, and write down why the result did or did not match your prediction.

## Exit ticket

**You are ready to continue when:** you can explain why loop order and tiling change performance without changing the matrix product and verify a tiled implementation.

**Next:** 06 — Measuring performance correctly.
